In [ ]:
# 1. Install arc-agi runtime from offline competition wheelhouse
import os, sys, glob, subprocess

print("Installing arc-agi runtime from offline wheelhouse...")
wheel_dirs = glob.glob("/kaggle/input/**/arc_agi_3_wheels", recursive=True)
if wheel_dirs:
    wheel_dir = wheel_dirs[0]
    print(f"Found wheels directory: {wheel_dir}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", wheel_dir, "arc-agi", "python-dotenv", "pillow"], check=False)
else:
    print("arc_agi_3_wheels not found in /kaggle/input; dependencies should be pre-installed.")


In [ ]:
# 2. Cohezion ARC-AGI-3 Directed Affordance Rarity Agent with BlueQubit QUBO Tie-Breaking (v18)
import hashlib
import json
import logging
import os
import random
import sys
import time
import glob
from collections import deque
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Set
import numpy as np

import arc_agi
import arcengine
from arcengine import GameAction, GameState, FrameDataRaw

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("CohezionARC3")

# Precomputed BlueQubit Quantum Symmetry Prior (D4 Dihedral Group)
DEFAULT_QUANTUM_SYMMETRIES = {
    "identity": 0.3035,
    "rot90": 0.0625,
    "rot180": 0.0625,
    "rot270": 0.0625,
    "flip_horizontal": 0.0625,
    "flip_vertical": 0.0625,
    "transpose": 0.0625,
    "anti_transpose": 0.0625,
}

def load_quantum_distillation_prior() -> dict[str, Any]:
    """Load BlueQubit quantum state kernels & symmetry priors with offline fallback."""
    paths = glob.glob("/kaggle/input/**/quantum_prior.npz", recursive=True) + [
        "data/quantum_prior.npz",
        "quantum_prior.npz"
    ]
    for p in paths:
        if os.path.exists(p):
            try:
                data = np.load(p, allow_pickle=True)
                sym_dict = dict(data["symmetry_weights"])
                kernel = data["kernel_matrix"]
                logger.info(f"Loaded BlueQubit quantum prior from {p}")
                return {
                    "symmetry_weights": {str(k): float(v) for k, v in sym_dict.items()},
                    "kernel_matrix": kernel,
                }
            except Exception as e:
                logger.warning(f"Error loading {p}: {e}")
    logger.info("Using embedded BlueQubit quantum prior constants.")
    return {
        "symmetry_weights": DEFAULT_QUANTUM_SYMMETRIES,
        "kernel_matrix": np.eye(8),
    }

QUANTUM_PRIOR = load_quantum_distillation_prior()

def hash_frame(frame_layers: list) -> str:
    if not frame_layers:
        return ""
    return hashlib.md5(np.ascontiguousarray(frame_layers[0]).tobytes()).hexdigest()[:16]

def find_components(grid: np.ndarray) -> list[dict[str, Any]]:
    """Extract foreground connected components sorted by AutoHarness affordance score."""
    h, w = grid.shape
    visited = np.zeros((h, w), dtype=bool)
    components = []
    vals, counts = np.unique(grid, return_counts=True)
    bg_color = vals[np.argmax(counts)]
    color_freq = dict(zip(vals, counts))

    for r in range(h):
        for c in range(w):
            if visited[r, c] or grid[r, c] == bg_color:
                continue
            color = int(grid[r, c])
            q = deque([(r, c)])
            visited[r, c] = True
            pixels = []
            while q:
                cr, cc = q.popleft()
                pixels.append((cr, cc))
                for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    nr, nc = cr + dr, cc + dc
                    if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc] and grid[nr, nc] == color:
                        visited[nr, nc] = True
                        q.append((nr, nc))
            rs = [p[0] for p in pixels]
            cs = [p[1] for p in pixels]
            r_min, r_max = min(rs), max(rs)
            c_min, c_max = min(cs), max(cs)
            size = len(pixels)
            color_cnt = int(color_freq[color])

            # AutoHarness Affordance Invariant:
            # Detect compact geometric features (singletons, bounding frames, keys)
            compactness = size / max(1, (r_max - r_min + 1) * (c_max - c_min + 1))
            dist_to_boundary = min(r_min, h - 1 - r_max, c_min, w - 1 - c_max)

            # Affordance rank: lowest color frequency first, smallest size, highest compactness
            affordance_rank = (color_cnt, size, -compactness, dist_to_boundary)

            components.append({
                "color": color,
                "color_count": color_cnt,
                "size": size,
                "compactness": compactness,
                "center_r": int(sum(rs) / len(rs)),
                "center_c": int(sum(cs) / len(cs)),
                "bbox": (r_min, c_min, r_max, c_max),
                "affordance_rank": affordance_rank,
            })

    components.sort(key=lambda c: c["affordance_rank"])
    return components

class CohezionDirectedRarityAgent:
    """Affordance-targeted search agent optimizing quadratic action economy ((human/ai)^2 score).
    
    Integrates:
    - Controllable avatar tracking via frame differential.
    - Direct vector steering toward rarest goal affordance.
    - BlueQubit QUBO action tie-breaking to prevent 2-step oscillation loops.
    - Affordance bounding-box jitter to escape local click minima.
    """
    def __init__(self, game_id: str, seed: int = 42):
        self.game_id = game_id
        self.rng = random.Random(seed + hash(game_id) % 100000)
        self.transition_model: dict[str, dict[int, str]] = {}
        self.state_visits: dict[str, int] = {}
        self.clicked_targets: dict[str, set[tuple[int, int]]] = {}
        self.last_state_hash: Optional[str] = None
        self.last_action_id: Optional[int] = None
        self.last_frame: Optional[np.ndarray] = None
        self.player_pos: Optional[tuple[int, int]] = None
        self.step_count = 0
        self.consecutive_loops = 0
        self.anti_oscillation_pairs = {(1, 2), (2, 1), (3, 4), (4, 3)}

    def _qubo_tie_break(self, candidate_actions: list[int], target_vec: Optional[tuple[int, int]]) -> int:
        """Select best action among candidates using QUBO energy minimization."""
        if len(candidate_actions) == 1:
            return candidate_actions[0]

        best_score = -float("inf")
        best_action = candidate_actions[0]
        dr, dc = target_vec if target_vec is not None else (0, 0)

        for act in candidate_actions:
            score = 0.0

            # Linear field h_a: alignment with target direction
            if act == 1 and dr < 0:
                score += 3.0 * abs(dr) / max(1, abs(dr) + abs(dc))
            elif act == 2 and dr > 0:
                score += 3.0 * abs(dr) / max(1, abs(dr) + abs(dc))
            elif act == 3 and dc < 0:
                score += 3.0 * abs(dc) / max(1, abs(dr) + abs(dc))
            elif act == 4 and dc > 0:
                score += 3.0 * abs(dc) / max(1, abs(dr) + abs(dc))

            # Quadratic coupling J_ab: heavy penalty for reversing immediately
            if self.last_action_id is not None and (self.last_action_id, act) in self.anti_oscillation_pairs:
                score -= 5.0

            # BlueQubit symmetry prior bonus
            if act in (1, 2):
                score += QUANTUM_PRIOR["symmetry_weights"].get("flip_vertical", 0.0) * 2.0
            elif act in (3, 4):
                score += QUANTUM_PRIOR["symmetry_weights"].get("flip_horizontal", 0.0) * 2.0

            # Tie-break noise
            score += self.rng.uniform(0.0, 0.01)

            if score > best_score:
                best_score = score
                best_action = act

        return best_action

    def choose_action(self, obs: FrameDataRaw) -> tuple[GameAction, Optional[dict[str, Any]]]:
        self.step_count += 1
        curr_hash = hash_frame(obs.frame)
        self.state_visits[curr_hash] = self.state_visits.get(curr_hash, 0) + 1
        grid = obs.frame[0] if obs.frame else None

        # Track avatar position from frame delta on directional movement
        if self.last_frame is not None and grid is not None and self.last_action_id in (1, 2, 3, 4):
            diff = (grid != self.last_frame)
            if 0 < np.sum(diff) <= 64:
                coords = np.argwhere(diff)
                self.player_pos = (int(np.mean(coords[:, 0])), int(np.mean(coords[:, 1])))

        self.last_frame = grid.copy() if grid is not None else None

        # Track state transitions and consecutive self-loops (wall collisions)
        if self.last_state_hash is not None and self.last_action_id is not None:
            if self.last_state_hash not in self.transition_model:
                self.transition_model[self.last_state_hash] = {}
            self.transition_model[self.last_state_hash][self.last_action_id] = curr_hash
            if curr_hash == self.last_state_hash:
                self.consecutive_loops += 1
            else:
                self.consecutive_loops = 0

        avail = obs.available_actions
        simple_ids = [a for a in avail if a not in (0, 6, 7)]
        has_click = 6 in avail

        if curr_hash not in self.clicked_targets:
            self.clicked_targets[curr_hash] = set()

        comps = find_components(grid) if grid is not None else []

        target_vec: Optional[tuple[int, int]] = None
        if self.player_pos is not None and comps:
            pr, pc = self.player_pos
            target = comps[0]
            target_vec = (target["center_r"] - pr, target["center_c"] - pc)

        # 1. Click handling (Preserve v14/v15 proven logic + jitter fallback)
        if has_click and (not simple_ids or self.consecutive_loops >= 2 or self.rng.random() < 0.40):
            unclicked = [c for c in comps if (c["center_c"], c["center_r"]) not in self.clicked_targets[curr_hash]]
            if unclicked:
                target = unclicked[0]
                coord = (target["center_c"], target["center_r"])
                self.clicked_targets[curr_hash].add(coord)
                self.last_state_hash = curr_hash
                self.last_action_id = 6
                return GameAction.ACTION6, {"x": coord[0], "y": coord[1]}
            elif comps and not simple_ids:
                target = comps[0] if len(comps) == 1 else self.rng.choice(comps)
                r_min, c_min, r_max, c_max = target["bbox"]
                coord = (self.rng.randint(c_min, c_max), self.rng.randint(r_min, r_max))
                self.last_state_hash = curr_hash
                self.last_action_id = 6
                return GameAction.ACTION6, {"x": coord[0], "y": coord[1]}

        # 2. Directed Movement handling with BlueQubit QUBO Tie-Breaking
        if simple_ids:
            known = self.transition_model.get(curr_hash, {})
            if target_vec is not None:
                dr, dc = target_vec
                if abs(dr) > abs(dc):
                    preferred = [1 if dr < 0 else 2, 3 if dc < 0 else 4]
                else:
                    preferred = [3 if dc < 0 else 4, 1 if dr < 0 else 2]
                
                viable_preferred = [act_id for act_id in preferred if act_id in simple_ids and known.get(act_id) != curr_hash]
                if viable_preferred:
                    chosen_id = self._qubo_tie_break(viable_preferred, target_vec)
                    self.last_state_hash = curr_hash
                    self.last_action_id = chosen_id
                    return GameAction.from_id(chosen_id), None

            untried = [a for a in simple_ids if a not in known]
            if untried:
                chosen_id = self._qubo_tie_break(untried, target_vec)
            else:
                non_loop = [a for a in simple_ids if known.get(a) != curr_hash]
                if non_loop:
                    min_visits = min(self.state_visits.get(known.get(a, ""), 0) for a in non_loop)
                    tied_actions = [a for a in non_loop if self.state_visits.get(known.get(a, ""), 0) == min_visits]
                    chosen_id = self._qubo_tie_break(tied_actions, target_vec)
                else:
                    chosen_id = self._qubo_tie_break(simple_ids, target_vec)

            self.last_state_hash = curr_hash
            self.last_action_id = chosen_id
            return GameAction.from_id(chosen_id), None

        fallback_id = avail[0] if avail else 0
        return GameAction.from_id(fallback_id), None

# Backward compatibility aliases
CohezionRaritySearchAgent = CohezionDirectedRarityAgent
CohezionGoExploreAgent = CohezionDirectedRarityAgent


In [ ]:
# 3. Competition Arcade Master Orchestrator
import os, sys, glob, time, urllib.request
from pathlib import Path

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
print(f"Cohezion ARC-AGI-3 Master Orchestrator: TRUE_SUBMISSION={TRUE_SUBMISSION}")

GATEWAY_URL = os.environ.get("ARC_BASE_URL", "http://gateway:8001/")
MAX_TOTAL_RUNTIME_S = 8.5 * 3600
START_TIME = time.time()

def wait_for_gateway(base_url: str, max_wait_s: int = 300) -> bool:
    print(f"Polling gateway at {base_url}api/games (up to {max_wait_s}s)...", flush=True)
    deadline = time.time() + max_wait_s
    while time.time() < deadline:
        try:
            req = urllib.request.Request(f"{base_url}api/games")
            with urllib.request.urlopen(req, timeout=5) as resp:
                if resp.status == 200:
                    print("✓ Gateway is online and ready!", flush=True)
                    return True
        except Exception:
            pass
        time.sleep(3)
    print("Warning: Gateway polling timed out.", flush=True)
    return False

if TRUE_SUBMISSION:
    wait_for_gateway(GATEWAY_URL)
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=GATEWAY_URL,
        arc_api_key="test-key-123",
    )
else:
    env_dirs = glob.glob("/kaggle/input/**/environment_files", recursive=True)
    env_dir = env_dirs[0] if env_dirs else "data/arc_prize/environment_files"
    print(f"Running in OFFLINE mode using environments from: {env_dir}")
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )

card_id = arcade.create_scorecard()
print(f"Opened Scorecard: {card_id}")

envs = arcade.get_environments()
print(f"Discovered {len(envs)} environments to solve.")

# In offline commit mode, run 2 environments quickly to pass the commit gate
if not TRUE_SUBMISSION:
    envs = envs[:2]

for idx, env_info in enumerate(envs):
    elapsed = time.time() - START_TIME
    if elapsed > MAX_TOTAL_RUNTIME_S:
        print(f"Time budget reached ({elapsed:.1f}s). Finalizing scorecard.")
        break

    game_id = env_info.game_id
    print(f"[{idx+1}/{len(envs)}] Playing {game_id} (Elapsed: {elapsed:.1f}s)...", flush=True)
    try:
        env = arcade.make(game_id, scorecard_id=card_id)
        if env is None:
            continue
        agent = CohezionGoExploreAgent(game_id, seed=idx * 100)
        obs = env.reset()

        step = 0
        while obs and obs.state not in (GameState.WIN, GameState.GAME_OVER) and step < 150:
            act, data = agent.choose_action(obs)
            obs = env.step(act, data=data)
            step += 1

        print(f"  -> {game_id} finished in {step} steps. State: {obs.state if obs else None}, Levels: {obs.levels_completed if obs else 0}")
    except Exception as exc:
        print(f"  Error on {game_id}: {exc}")

sc = arcade.close_scorecard(card_id)
print(f"✓ Closed Scorecard {card_id}. Total actions: {sc.total_actions if sc else 0}, Completed: {sc.total_levels_completed if sc else 0}")


In [ ]:
# 4. Commit Validator Parquet Handler
# During commit (not TRUE_SUBMISSION), generates initial valid parquet so Kaggle enables Submit.
# During competition rerun (TRUE_SUBMISSION), Kaggle generates submission.parquet automatically from the scorecard.
import os
import pandas as pd
from pathlib import Path

target_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
target_file = target_dir / "submission.parquet"

if not TRUE_SUBMISSION:
    print("Generating commit-time validator submission.parquet...")
    commit_df = pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"]
    )
    commit_df.to_parquet(target_file, index=False)
    print(f"✓ Commit validator submission.parquet written ({target_file.stat().st_size} bytes).")
else:
    print("TRUE_SUBMISSION rerun complete. Gateway recorded scorecard for evaluation.")
